# 01 — Synthetic Japanese Insurance Data (rebuilt engine)

Data-driven generation using **real datasets** and a **correlated, lineage-tracked dirty layer**:

| Need | Source |
|---|---|
| Person names (kanji + demographics) | NVIDIA **Nemotron-Personas-Japan** (1M personas; real surname/given frequencies, sex, age, prefecture) |
| Kana / romaji | Derived from the kanji via pykakasi (guaranteed orthographic consistency) |
| Postal codes + towns | Official **Japan Post** ken_all (118k towns, all 47 prefectures) |
| Company names | **EDINET** (4,253 real legal names) |
| Building names | Composer over a real lexicon (available in every prefecture) |
| Dirty data | Correlated **events** (rename/move/contact-stale, paper-OCR bursts, legacy format incl. era dates + half-width, correlated missing), scaled by quality band × agency factors |

The engine lives in the `synthetic/` package (SRP: providers / noise / truth / engine; OCP: new noise types register as events). __This notebook is a thin driver + verification__.

Outputs are exactly `nayose.ipynb`'s contract: visible `reference_records` + `incoming_queries`, hidden `record_truth` / `party_truth` / `relation_truth` / `query_truth` / `evaluation_truth` / `field_lineage`, plus `agencies` and a `generation_manifest.json`.

> Data sources note: personas © NVIDIA Nemotron (CC-BY 4.0), postal data © Japan Post, company names © EDINET — regenerable, cached under `data/`.

In [ ]:
# 1. Dependencies and runtime preflight
from pathlib import Path
import json, os, sys, time

import numpy as np
import pandas as pd

from synthetic import (
    AddressSource, BuildingComposer, CompanySource, NoiseConfig,
    PersonaSource, SyntheticConfig, generate_agencies,
)
from synthetic.pipeline import generate, save_all

print("Python:", sys.version.split()[0])
print("Engine: synthetic package (see pyproject.toml for pinned tooling).")
print("Preflight complete.")

In [ ]:
# 2. Configuration (env-overridable, mirrors NAYOSE_*)
CFG = SyntheticConfig()
NOISE = NoiseConfig()
rng = np.random.default_rng(CFG.seed)
CFG.output_dir.mkdir(parents=True, exist_ok=True)

print("Resolved run:")
print(json.dumps({
    "seed": CFG.seed, "profile": CFG.profile, "difficulty": CFG.difficulty,
    "n_parties": CFG.n_parties, "mean_records": CFG.mean_records,
    "poor": CFG.poor, "catastrophic": CFG.catastrophic, "stale": CFG.stale,
    "output_dir": str(CFG.output_dir),
}, indent=2))

In [ ]:
# 3. Real-data providers (downloaded once, cached under data/)
persona = PersonaSource(CFG.raw_dir)
companies = CompanySource()
address = AddressSource(CFG.raw_dir)
buildings = BuildingComposer()

t0 = time.perf_counter()
agencies = generate_agencies(CFG, rng)
print(f"agencies: {len(agencies):,} (providers ready in {time.perf_counter()-t0:.1f}s)")

In [ ]:
# 4. Generate everything (parties -> observations -> queries -> six truth tables)
from synthetic.pipeline import generate, save_all

t0 = time.perf_counter()
RESULT = generate(CFG, rng)
print(f"pipeline generated in {time.perf_counter()-t0:.1f}s")
RESULT.parties.head()

In [ ]:
# 5. Dirty observations + benchmark queries (display)
fld = ["name_kanji","name_kana","name_romaji","date_birth","postal_code","address","phone","email"]
print(f"observations: {len(RESULT.observations):,} "
      f"({len(RESULT.observations)/len(RESULT.parties):.2f} records/party)")
print("missing cell rate:", f"{RESULT.observations[fld].eq('').mean().mean():.1%}")
RESULT.observations.head()
print(f"queries: {len(RESULT.queries):,}")
RESULT.queries.head()

In [ ]:
# 6. Ground-truth tables (six hidden tables, exact nayose contract)
for stem, frame in RESULT.ground_truth.items():
    print(f"  {stem:<20} {len(frame):>9,} rows")

In [ ]:
# 7. Save all tables (visible + hidden) + manifest (atomic parquet writes)
save_all(CFG, RESULT)
print("\n=== Data generation complete ===")

In [ ]:
# 8. Contract verification (the nayose.ipynb gate, enforced here too)
forbidden = {"truth_id", "target_truth_id", "relation_id", "scenario", "hard_case",
             "split", "record_quality", "transformation"}
assert set(RESULT.ground_truth["record_truth"].record_id) == set(RESULT.observations.record_id)
assert set(RESULT.ground_truth["evaluation_truth"].query_id) == set(RESULT.queries.query_id)
assert set(RESULT.ground_truth["query_truth"].query_id) == set(RESULT.queries.query_id)
assert not forbidden & set(RESULT.observations.columns)
assert not forbidden & set(RESULT.queries.columns)
assert len(RESULT.ground_truth["field_lineage"]) == (len(RESULT.observations) + len(RESULT.queries)) * 8
assert set(RESULT.ground_truth["evaluation_truth"].scenario) >= set(CFG.query_mix)

print("Visible snapshots carry no hidden truth; ids match exactly; lineage covers every field.")
print("Split shares:", dict(RESULT.ground_truth["record_truth"].split.value_counts(normalize=True).round(3)))

In [ ]:
# 9. Contract verification (the nayose.ipynb gate, enforced here too)
forbidden = {"truth_id", "target_truth_id", "relation_id", "scenario", "hard_case",
             "split", "record_quality", "transformation"}
assert set(RESULT.ground_truth["record_truth"].record_id) == set(RESULT.observations.record_id)
assert set(RESULT.ground_truth["evaluation_truth"].query_id) == set(RESULT.queries.query_id)
assert set(RESULT.ground_truth["query_truth"].query_id) == set(RESULT.queries.query_id)
assert not forbidden & set(RESULT.observations.columns)
assert not forbidden & set(RESULT.queries.columns)
assert len(RESULT.ground_truth["field_lineage"]) == (len(RESULT.observations) + len(RESULT.queries)) * 8
assert set(RESULT.ground_truth["evaluation_truth"].scenario) >= set(CFG.query_mix)

print("Visible snapshots carry no hidden truth; ids match exactly; lineage covers every field.")
print("Split shares:", dict(RESULT.ground_truth["record_truth"].split.value_counts(normalize=True).round(3)))

## 10. Run the matcher in tandem

```bash
# both notebooks default to data/ (NAYOSE_DATA_DIR override available)
jupyter nbconvert --to notebook --execute synthetic_data.ipynb
NAYOSE_MODEL_FAMILY=off NAYOSE_DEVICE=cpu jupyter nbconvert --to notebook --execute nayose.ipynb
```

`nayose.ipynb` reads `reference_records`, `incoming_queries`, the six `ground_truth/*` tables and the manifest — all produced above. Safe caps: use the `smoke`/`mac` profiles for quick or 16 GB-Mac runs.